In [ ]:
import os
import pandas as pd
import re
import numpy as np
from dotenv import load_dotenv
import os
load_dotenv()  
pd.set_option('display.max_rows', None)


''' To upload SG files do the following: 
1. go to pga.com site and go to stats site, then to the SG:Total 
2. Save the file as SG_[tourneyID]_[year].csv 
3. Save the file to the /Users/brandon/Desktop/Golf_Data/upload_staging/data_for_upload/ folder 

Other info: 
4. the process should create a merged file (this can be deleted at some point) in the folder 

'''

# Base folder path
SG_CSV_PATH = os.getenv("SG_CSV_PATH")
JOIN_FILE_CSV_PATH = os.getenv("JOIN_FILE_CSV_PATH")


# Load the file to join
main_df = pd.read_csv(JOIN_FILE_CSV_PATH)
main_df = main_df[['TourneyID', 'Tournament', 'Week_Of_Season', 'Course', 'Signature_Event', 'Major_Event', 'par']]

# --- Parse ID and Year from filename ---
def parse_id_and_year(filename):
    match = re.match(r'.*_(\d+)_(\d{4})\.csv', filename)
    if match:
        return match.group(1), match.group(2)
    return None, None


# --- Identify Leader Files ---
leader_files = [
    f for f in os.listdir(SG_CSV_PATH)
    if f.startswith('sg') and f.endswith('.csv')
]

In [2]:
# --- Process Each Leader File ---
for leader_file in leader_files:
    tourney_id, year = parse_id_and_year(leader_file)
    if not tourney_id:
        continue

    print(f"Processing SG file: {leader_file} (TourneyID={tourney_id}, year={year})")

    # Load leader file
    leader_path = os.path.join(SG_CSV_PATH, leader_file)
    leader_df = pd.read_csv(leader_path)
    leader_df['TourneyID'] = int(tourney_id)
    leader_df['year'] = year

    # Join metadata (left join on tourneyID)
    enriched_df = leader_df.merge(main_df, on='TourneyID', how='left')

    # Remove blank rows
    enriched_df.dropna(how='all', inplace=True)

    # Trim spaces from column names in join file
    enriched_df.columns = enriched_df.columns.str.strip()

    # Trim spaces from string values in join file
    enriched_df = enriched_df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)



    # Save output


Processing SG file: sg_23_2026.csv (TourneyID=23, year=2026)


In [3]:
tourney_file_name = enriched_df['Tournament'].unique().tolist()
print(tourney_file_name)

['THE PLAYERS Championship']


In [ ]:
# Need to re-do postgres creds 
from dotenv import load_dotenv


# Load environment variables from .env file
load_dotenv()

# Database configuration from environment variables
db_config = {
    'host': os.getenv('DB_HOST'),
    'database': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'port': os.getenv('DB_PORT', '5432')
}
schema = os.getenv('DB_SCHEMA')

# Test the connection
try:
    conn = psycopg2.connect(**db_config)
    conn.close()
    print("✅ Database connection successful!")
    print(f"   Connected to: {db_config['database']} on {db_config['host']}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Database connection successful!
   Connected to: brdb on localhost


In [6]:
# Target table name
table_name = 'sg_data'

In [7]:
#adding 2 null cols for Movement and PlayerID 
enriched_df['MOVEMENT'] = np.nan
#enriched_df['PLAYER_ID'] = np.nan
enriched_df['PLAYER_ID'] = None
#enriched_df['PLAYER_ID'] = pd.to_numeric(enriched_df['PLAYER_ID'], errors='coerce').astype('Int64')
#enriched_df = enriched_df.where(pd.notna(enriched_df), None)


#enriched_df['MOVEMENT'] = pd.Series([np.nan] * len(enriched_df), dtype='Int64')  # note the capital 'I'
#enriched_df['PLAYER_ID'] = pd.Series([np.nan] * len(enriched_df), dtype='Int64')  # note the capital 'I'


# reorder cols 
enriched_df = enriched_df[['Rank', 'MOVEMENT', 'PLAYER_ID', 'Player','Avg', 'Total SG:T', 'Total SG:T2G'
                           , 'Total SG:P', 'Measured Rounds','TourneyID','year','Tournament'
                           , 'Week_Of_Season' , 'Course','Signature_Event','Major_Event','par']]




In [8]:
enriched_df

,Rank,MOVEMENT,PLAYER_ID,Player,Avg,Total SG:T,Total SG:T2G,Total SG:P,Measured Rounds,TourneyID,year,Tournament,Week_Of_Season,Course,Signature_Event,Major_Event,par
0,1,NaN,None,Cameron Young,3.663,14.653,9.840,4.813,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
1,2,NaN,None,Matt Fitzpatrick,3.413,13.653,12.171,1.482,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
2,3,NaN,None,Xander Schauffele,3.163,12.653,9.989,2.664,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
3,4,NaN,None,Robert MacIntyre,2.913,11.653,7.065,4.588,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
4,5,NaN,None,Ludvig Åberg,2.663,10.653,9.234,1.419,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
5,5,NaN,None,Jacob Bridgeman,2.663,10.653,0.902,9.751,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
6,5,NaN,None,Sudarshan Yellamaraju,2.663,10.653,5.152,5.501,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
7,8,NaN,None,Tommy Fleetwood,2.413,9.653,12.391,-2.738,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
8,8,NaN,None,Sepp Straka,2.413,9.653,7.090,2.563,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
9,8,NaN,None,Justin Thomas,2.413,9.653,5.385,4.268,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0


In [9]:
df = enriched_df.rename(columns={'Rank': 'Rank', 'MOVEMENT': 'Movement', 
                            'PLAYER_ID': 'PlayerID', 'Player': 'Player', 'Total SG:T': 'Total_SG_T'
                            , 'Total SG:T2G': 'Total_SG_T2G', 'Total SG:P':'Total_SG_P'
                            , 'Measured Rounds':'Measured_Rounds' })

df = df.dropna(subset=df.columns[:7], how='all')

columns_to_clean = ['par', 'Week_Of_Season', 'year','Rank','Measured_Rounds'] # removing Player_ID since its null/blank



#print(df.head())

In [10]:
df

,Rank,Movement,PlayerID,Player,Avg,Total_SG_T,Total_SG_T2G,Total_SG_P,Measured_Rounds,TourneyID,year,Tournament,Week_Of_Season,Course,Signature_Event,Major_Event,par
0,1,NaN,None,Cameron Young,3.663,14.653,9.840,4.813,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
1,2,NaN,None,Matt Fitzpatrick,3.413,13.653,12.171,1.482,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
2,3,NaN,None,Xander Schauffele,3.163,12.653,9.989,2.664,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
3,4,NaN,None,Robert MacIntyre,2.913,11.653,7.065,4.588,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
4,5,NaN,None,Ludvig Åberg,2.663,10.653,9.234,1.419,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
5,5,NaN,None,Jacob Bridgeman,2.663,10.653,0.902,9.751,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
6,5,NaN,None,Sudarshan Yellamaraju,2.663,10.653,5.152,5.501,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
7,8,NaN,None,Tommy Fleetwood,2.413,9.653,12.391,-2.738,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
8,8,NaN,None,Sepp Straka,2.413,9.653,7.090,2.563,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0
9,8,NaN,None,Justin Thomas,2.413,9.653,5.385,4.268,4,23,2026,THE PLAYERS Championship,11.0,TPC Sawgrass,N,N,72.0


In [11]:
df.drop(columns=['TourneyID'], inplace=True)

In [12]:
for col in columns_to_clean:
    df[col] = df[col].apply(lambda x: None if pd.isna(x) else int(x))

In [ ]:
#df = df.dropna(subset=['pos'])
#df = df.dropna(how='all')

In [ ]:
df

In [13]:
conn = psycopg2.connect(**db_config)
cur = conn.cursor()
for _, row in df.iterrows():
    columns = ', '.join(df.columns)
    placeholders = ', '.join(['%s'] * len(row))
    insert_sql = f"INSERT INTO {schema}.{table_name} ({columns}) VALUES ({placeholders})"
    
    # DEBUG: print the row and data types
    print("Row data:", row.to_dict())

    try:
        cur.execute(insert_sql, tuple(row))
    except Exception as e:
        print("Error on row:", row.to_dict())
        print("Data types:", [type(x) for x in row])
        raise e  # re-raise to stop execution

    #cur.execute(insert_sql, tuple(row))
        
    conn.commit()
    print(f"Inserted data from {tourney_file_name} into {table_name}")

Row data: {'Rank': 1, 'Movement': nan, 'PlayerID': None, 'Player': 'Cameron Young', 'Avg': 3.663, 'Total_SG_T': 14.653, 'Total_SG_T2G': 9.84, 'Total_SG_P': 4.813, 'Measured_Rounds': 4, 'year': 2026, 'Tournament': 'THE PLAYERS Championship', 'Week_Of_Season': 11, 'Course': 'TPC Sawgrass', 'Signature_Event': 'N', 'Major_Event': 'N', 'par': 72}
Inserted data from ['THE PLAYERS Championship'] into sg_data
Row data: {'Rank': 2, 'Movement': nan, 'PlayerID': None, 'Player': 'Matt Fitzpatrick', 'Avg': 3.413, 'Total_SG_T': 13.653, 'Total_SG_T2G': 12.171, 'Total_SG_P': 1.482, 'Measured_Rounds': 4, 'year': 2026, 'Tournament': 'THE PLAYERS Championship', 'Week_Of_Season': 11, 'Course': 'TPC Sawgrass', 'Signature_Event': 'N', 'Major_Event': 'N', 'par': 72}
Inserted data from ['THE PLAYERS Championship'] into sg_data
Row data: {'Rank': 3, 'Movement': nan, 'PlayerID': None, 'Player': 'Xander Schauffele', 'Avg': 3.163, 'Total_SG_T': 12.653, 'Total_SG_T2G': 9.989, 'Total_SG_P': 2.664, 'Measured_Rounds'